In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import timedelta
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

In [2]:
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#64748b',
    'ytick.color': '#64748b',
    'text.color': '#e2e8f0',
    'grid.color': '#1e293b',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': '#f1f5f9',
})

BLUE   = '#3b82f6'
TEAL   = '#14b8a6'
AMBER  = '#f59e0b'
RED    = '#ef4444'
GREEN  = '#22c55e'
PURPLE = '#a855f7'
COLORS = [BLUE, TEAL, AMBER, GREEN, PURPLE, RED, '#ec4899', '#f97316']

import os
os.makedirs('/home/claude/project/outputs/charts', exist_ok=True)

In [5]:
orders = pd.read_csv('../data/orders.csv',       parse_dates=['order_date', 'signup_date'])
users  = pd.read_csv('../data/users.csv',        parse_dates=['signup_date'])
funnel = pd.read_csv('../data/funnel_events.csv', parse_dates=['date'])

orders['month']     = orders['order_date'].dt.to_period('M')
orders['hour']      = orders['order_date'].dt.hour
orders['dayofweek'] = orders['order_date'].dt.day_name()

print("=" * 60)
print("   FOOD DELIVERY PRODUCT ANALYTICS — EXECUTIVE SUMMARY")
print("=" * 60)


   FOOD DELIVERY PRODUCT ANALYTICS — EXECUTIVE SUMMARY


In [6]:
stages      = ['App Open', 'Search', 'Restaurant View', 'Add to Cart', 'Checkout', 'Order Placed']
stage_keys  = ['app_open', 'search', 'restaurant_view', 'add_to_cart', 'checkout', 'order_placed']
counts      = [funnel[funnel['stage'] == s]['session_id'].nunique() for s in stage_keys]
drops       = [0] + [round((1 - counts[i]/counts[i-1])*100, 1) for i in range(1, len(counts))]
conversion  = round(counts[-1]/counts[0]*100, 2)

print(f"\n📊 FUNNEL ANALYSIS")
print(f"   Overall Conversion Rate: {conversion}%")
for s, c, d in zip(stages, counts, drops):
    print(f"   {s:<20} {c:>7,}  {'↓ ' + str(d) + '% drop' if d > 0 else ''}")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(stages[::-1], counts[::-1], color=BLUE, alpha=0.85, height=0.6)
for bar, count, drop in zip(bars, counts[::-1], drops[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', color='#94a3b8', fontsize=10)
    if drop > 0:
        ax.text(bar.get_width()/2, bar.get_y() + bar.get_height()/2,
                f'-{drop}%', va='center', ha='center', color='white', fontsize=9, fontweight='bold')
ax.set_title('User Funnel — App Open to Order Placed', pad=15)
ax.set_xlabel('Sessions')
ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/01_funnel.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("   ✅ Chart saved: 01_funnel.png")


📊 FUNNEL ANALYSIS
   Overall Conversion Rate: 0.94%
   App Open              50,000  
   Search                36,089  ↓ 27.8% drop
   Restaurant View       20,982  ↓ 41.9% drop
   Add to Cart            8,675  ↓ 58.7% drop
   Checkout               2,438  ↓ 71.9% drop
   Order Placed             472  ↓ 80.6% drop
   ✅ Chart saved: 01_funnel.png


In [7]:
o          = orders[orders['cancelled'] == 0].sort_values('order_date')
user_first = o.groupby('user_id')['order_date'].min().reset_index()
user_first.columns = ['user_id', 'first_order_date']
o          = o.merge(user_first, on='user_id')
o['cohort_month']  = o['first_order_date'].dt.to_period('M')
o['order_month']   = o['order_date'].dt.to_period('M')
o['period_number'] = (o['order_month'] - o['cohort_month']).apply(lambda x: x.n)

cohort_data  = o.groupby(['cohort_month', 'period_number'])['user_id'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='cohort_month', columns='period_number', values='user_id')
retention    = (cohort_pivot.divide(cohort_pivot[0], axis=0) * 100).round(1).iloc[:9, :7]

print(f"\n📈 RETENTION ANALYSIS")
print(f"   Avg Month-1 Retention: {retention[1].mean():.1f}%")
print(f"   Avg Month-3 Retention: {retention[3].mean():.1f}%")

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(retention, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, linecolor='#0f172a', ax=ax,
            annot_kws={'size': 9})
ax.set_title('Monthly Cohort Retention Heatmap (%)', pad=15)
ax.set_xlabel('Months Since First Order')
ax.set_ylabel('Cohort Month')
plt.tight_layout()
plt.savefig('../outputs/charts/02_retention_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("   ✅ Chart saved: 02_retention_heatmap.png")


📈 RETENTION ANALYSIS
   Avg Month-1 Retention: 21.6%
   Avg Month-3 Retention: 20.7%
   ✅ Chart saved: 02_retention_heatmap.png


In [8]:
# ── 3. USER SEGMENTATION ──────────────────────────────────────────────────────
active     = orders[orders['cancelled'] == 0]
max_date   = orders['order_date'].max()
user_stats = active.groupby('user_id').agg(
    total_orders=('order_id', 'count'),
    total_spend=('order_value', 'sum'),
    last_order=('order_date', 'max')
).reset_index()
user_stats['days_since_last'] = (max_date - user_stats['last_order']).dt.days

def segment(row):
    if row['total_orders'] >= 20 and row['days_since_last'] <= 30:   return 'Power User'
    elif row['total_orders'] >= 8 and row['days_since_last'] <= 60:  return 'Regular User'
    elif row['days_since_last'] > 90:                                 return 'Churned'
    elif row['total_orders'] <= 2:                                    return 'New User'
    else:                                                              return 'Occasional User'

user_stats['segment'] = user_stats.apply(segment, axis=1)
seg_counts = user_stats['segment'].value_counts()
seg_spend  = user_stats.groupby('segment')['total_spend'].mean().round(0)

print(f"\n👥 USER SEGMENTATION")
for seg in seg_counts.index:
    print(f"   {seg:<20} {seg_counts[seg]:>5} users  |  Avg spend ₹{seg_spend[seg]:,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
seg_order = ['Power User', 'Regular User', 'Occasional User', 'New User', 'Churned']
seg_c = [seg_counts.get(s, 0) for s in seg_order]
seg_s = [seg_spend.get(s, 0)  for s in seg_order]

axes[0].pie(seg_c, labels=seg_order, colors=COLORS[:5], autopct='%1.1f%%',
            startangle=90, textprops={'color': '#e2e8f0', 'fontsize': 10})
axes[0].set_title('User Segment Distribution')

bars = axes[1].bar(seg_order, seg_s, color=COLORS[:5], alpha=0.85, width=0.6)
for bar, val in zip(bars, seg_s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'₹{val:,.0f}', ha='center', color='#94a3b8', fontsize=9)
axes[1].set_title('Average Total Spend by Segment (₹)')
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('../outputs/charts/03_user_segments.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("   ✅ Chart saved: 03_user_segments.png")


👥 USER SEGMENTATION
   Churned               1874 users  |  Avg spend ₹620
   Occasional User        803 users  |  Avg spend ₹1,394
   New User               775 users  |  Avg spend ₹511
   Regular User           772 users  |  Avg spend ₹4,741
   Power User             150 users  |  Avg spend ₹6,843
   ✅ Chart saved: 03_user_segments.png


In [9]:
# ── 4. ORDER TRENDS ───────────────────────────────────────────────────────────
monthly   = active.groupby('month').agg(orders=('order_id','count'), revenue=('order_value','sum')).reset_index()
monthly['month_str'] = monthly['month'].astype(str)
hourly    = active.groupby('hour')['order_id'].count()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily     = active.groupby('dayofweek')['order_id'].count().reindex(day_order)

print(f"\n📅 ORDER TRENDS")
print(f"   Peak hour: {hourly.idxmax()}:00 ({hourly.max():,} orders)")
print(f"   Peak day:  {daily.idxmax()} ({daily.max():,} orders)")
print(f"   Total Revenue: ₹{active['order_value'].sum():,.0f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(monthly['month_str'], monthly['orders'], color=BLUE, linewidth=2.5, marker='o', markersize=5)
axes[0,0].fill_between(range(len(monthly)), monthly['orders'], alpha=0.15, color=BLUE)
axes[0,0].set_title('Monthly Orders')
axes[0,0].set_xticklabels(monthly['month_str'], rotation=45, fontsize=8)
axes[0,0].spines[['top','right']].set_visible(False)

axes[0,1].plot(monthly['month_str'], monthly['revenue']/1e6, color=GREEN, linewidth=2.5, marker='o', markersize=5)
axes[0,1].fill_between(range(len(monthly)), monthly['revenue']/1e6, alpha=0.15, color=GREEN)
axes[0,1].set_title('Monthly Revenue (₹ Millions)')
axes[0,1].set_xticklabels(monthly['month_str'], rotation=45, fontsize=8)
axes[0,1].spines[['top','right']].set_visible(False)

axes[1,0].bar(hourly.index, hourly.values, color=AMBER, alpha=0.85, width=0.8)
axes[1,0].set_title('Orders by Hour of Day')
axes[1,0].set_xlabel('Hour')
axes[1,0].spines[['top','right']].set_visible(False)

axes[1,1].bar(daily.index, daily.values, color=PURPLE, alpha=0.85, width=0.6)
axes[1,1].set_title('Orders by Day of Week')
axes[1,1].tick_params(axis='x', rotation=30)
axes[1,1].spines[['top','right']].set_visible(False)

plt.suptitle('Order Trends Analysis', fontsize=15, fontweight='bold', color='#f1f5f9', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/charts/04_order_trends.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("   ✅ Chart saved: 04_order_trends.png")



📅 ORDER TRENDS
   Peak hour: 4:00 (1,048 orders)
   Peak day:  Saturday (3,388 orders)
   Total Revenue: ₹7,363,993
   ✅ Chart saved: 04_order_trends.png


In [12]:
# ── 5. A/B TEST ───────────────────────────────────────────────────────────────
control   = active[active['promo_used'] == 'NONE']['order_value']
treatment = active[active['promo_used'] == 'FREEDELIVERY']['order_value']
t_stat, p_value = scipy_stats.ttest_ind(control, treatment)
uplift = ((treatment.mean() - control.mean()) / control.mean()) * 100

print(f"\n🧪 A/B TEST — FREE DELIVERY PROMO")
print(f"   Control avg:   ₹{control.mean():.2f}")
print(f"   Treatment avg: ₹{treatment.mean():.2f}")
print(f"   Uplift: {uplift:+.1f}%  |  p-value: {p_value:.4f}  {'✅ SIGNIFICANT' if p_value < 0.05 else '❌ NOT SIGNIFICANT'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
promo_stats = active.groupby('promo_used')['order_id'].count().sort_values(ascending=False)
axes[0].bar(promo_stats.index, promo_stats.values, color=COLORS[:len(promo_stats)], alpha=0.85, width=0.6)
axes[0].set_title('Orders by Promo Type')
axes[0].set_ylabel('Number of Orders')
axes[0].spines[['top','right']].set_visible(False)

axes[1].hist(control.sample(2000), bins=40, alpha=0.6, color=BLUE,  label=f'No Promo (₹{control.mean():.0f} avg)')
axes[1].hist(treatment.sample(min(2000, len(treatment))), bins=40, alpha=0.6, color=GREEN, label=f'Free Delivery (₹{treatment.mean():.0f} avg)')
axes[1].axvline(control.mean(),   color=BLUE,  linestyle='--', linewidth=2)
axes[1].axvline(treatment.mean(), color=GREEN, linestyle='--', linewidth=2)
axes[1].set_title(f'A/B Test: Order Value  |  Uplift: {uplift:+.1f}%  |  p={p_value:.4f}')
axes[1].set_xlabel('Order Value (₹)')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/05_ab_test.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("   ✅ Chart saved: 05_ab_test.png")



🧪 A/B TEST — FREE DELIVERY PROMO
   Control avg:   ₹320.49
   Treatment avg: ₹320.63
   Uplift: +0.0%  |  p-value: 0.9474  ❌ NOT SIGNIFICANT
   ✅ Chart saved: 05_ab_test.png


In [13]:
 #── 6. CUISINE & CITY ─────────────────────────────────────────────────────────
cuisine_stats = active.groupby('cuisine').agg(orders=('order_id','count'), avg_value=('order_value','mean')).reset_index().sort_values('orders', ascending=False)
city_stats    = active.groupby('city').agg(revenue=('order_value','sum')).reset_index().sort_values('revenue', ascending=False)

print(f"\n🍕 TOP CUISINES")
for _, row in cuisine_stats.head(4).iterrows():
    print(f"   {row['cuisine']:<20} {row['orders']:>5} orders  |  ₹{row['avg_value']:.0f} avg")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(cuisine_stats['cuisine'][::-1], cuisine_stats['orders'][::-1], color=COLORS[:len(cuisine_stats)], alpha=0.85, height=0.6)
axes[0].set_title('Orders by Cuisine')
axes[0].spines[['top','right','left']].set_visible(False)

axes[1].bar(city_stats['city'], city_stats['revenue']/1e6, color=COLORS[:len(city_stats)], alpha=0.85, width=0.6)
axes[1].set_title('Revenue by City (₹ Millions)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/06_cuisine_city.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("   ✅ Chart saved: 06_cuisine_city.png")


🍕 TOP CUISINES
   North Indian          5017 orders  |  ₹322 avg
   South Indian          3360 orders  |  ₹321 avg
   Biryani               3260 orders  |  ₹318 avg
   Chinese               3169 orders  |  ₹322 avg
   ✅ Chart saved: 06_cuisine_city.png


In [14]:
# ── 7. CANCELLATION & DELIVERY ────────────────────────────────────────────────
cancel_rate  = orders['cancelled'].mean() * 100
late_rate    = orders['late_delivery'].mean() * 100
avg_delivery = orders['delivery_time_mins'].mean()
late_rating  = orders[orders['late_delivery'] == 1]['rating'].mean()
ontime_rating= orders[orders['late_delivery'] == 0]['rating'].mean()

print(f"\n⚠️  CANCELLATION & DELIVERY")
print(f"   Cancellation Rate: {cancel_rate:.1f}%")
print(f"   Late Delivery Rate: {late_rate:.1f}%")
print(f"   Avg Delivery Time: {avg_delivery:.0f} mins")
print(f"   On-time Rating: ⭐{ontime_rating:.2f}  |  Late Rating: ⭐{late_rating:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cancel_city = orders.groupby('city')['cancelled'].mean().sort_values(ascending=False) * 100
axes[0].bar(cancel_city.index, cancel_city.values, color=RED, alpha=0.75, width=0.6)
axes[0].set_title('Cancellation Rate by City (%)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].spines[['top','right']].set_visible(False)

bp = axes[1].boxplot([orders[orders['late_delivery']==0]['rating'], orders[orders['late_delivery']==1]['rating']],
                      labels=['On-Time', 'Late'], patch_artist=True,
                      medianprops={'color': 'white', 'linewidth': 2})
bp['boxes'][0].set_facecolor(GREEN)
bp['boxes'][1].set_facecolor(RED)
axes[1].set_title(f'Rating: On-Time (⭐{ontime_rating:.2f}) vs Late (⭐{late_rating:.2f})')
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/charts/07_cancellation_delivery.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("   ✅ Chart saved: 07_cancellation_delivery.png")

print("\n" + "=" * 60)
print("✅ All analysis complete! Charts saved to ../outputs/charts/")
print("=" * 60)


⚠️  CANCELLATION & DELIVERY
   Cancellation Rate: 8.1%
   Late Delivery Rate: 15.0%
   Avg Delivery Time: 32 mins
   On-time Rating: ⭐4.13  |  Late Rating: ⭐4.16
   ✅ Chart saved: 07_cancellation_delivery.png

✅ All analysis complete! Charts saved to ../outputs/charts/


In [15]:
import os
print(os.getcwd())

C:\Users\shubh\product_analytics_project\notebooks
